In [1]:
#This cell is for creating folders to store the outputs in.
import os

base_path = "/kaggle/working"

folders = [
    "features",
    "checkpoints",
    "plots"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


# Yoga Pose Classification & Feedback System

This project builds an end-to-end pipeline for:
- Pose landmark extraction (MediaPipe)
- Pose classification (MLP)
- Pose correction feedback (angle-based analysis)

---

In [3]:
# Used to verify the dataset's root path. 
import os 

# Dataset Root Check 
ROOT = "/kaggle/input/datasets/kolliprasannaadarsh/manoj-dataset/Manoj-Dataset" 
print("ROOT exists:", os.path.exists(ROOT)) 
print("ROOT contents:", os.listdir(ROOT)) 

# dataset path
DATASET_ROOT = ROOT

# list pose folders
poses = [d for d in os.listdir(DATASET_ROOT)
         if os.path.isdir(os.path.join(DATASET_ROOT, d))]

print("Detected poses:", poses)
print("Total poses:", len(poses))

ROOT exists: True
ROOT contents: ['Vrukshasana', 'Triangle', 'Veerabhadrasana', 'BaddhaKonasana', 'Natarajasana', 'UtkataKonasana', 'Downward_dog', 'ArdhaChandrasana']
Detected poses: ['Vrukshasana', 'Triangle', 'Veerabhadrasana', 'BaddhaKonasana', 'Natarajasana', 'UtkataKonasana', 'Downward_dog', 'ArdhaChandrasana']
Total poses: 8


## Dataset Preparation
We organize images into structured labels and splits. This includes parsing directory structures, mapping class labels, and preparing the dataset for pose landmark extraction.

In [5]:
#This cell builds a dataframe for all images and labels. 
import os
import glob
import pandas as pd

rows = []

pose_folders = [
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
]

for pose in pose_folders:

    image_dir = os.path.join(DATASET_ROOT, pose, "Images")
    landmark_dir = os.path.join(DATASET_ROOT, pose, "Landmarks")

    images = (
        glob.glob(os.path.join(image_dir, "*.jpg")) +
        glob.glob(os.path.join(image_dir, "*.png")) +
        glob.glob(os.path.join(image_dir, "*.jpeg"))
    )

    for img_path in images:

        filename = os.path.basename(img_path)

        # try matching landmark file with same name
        possible_landmarks = glob.glob(
            os.path.join(landmark_dir, filename.split(".")[0] + ".*")
        )

        landmark_path = possible_landmarks[0] if possible_landmarks else None

        rows.append({
            "filepath": img_path,
            "pose": pose,
            "landmark_path": landmark_path
        })

df = pd.DataFrame(rows)

print("Total images:", len(df))
print("Total poses:", df["pose"].nunique())
print("Pose names:", sorted(df["pose"].unique()))

df.head()

Total images: 484
Total poses: 8
Pose names: ['ArdhaChandrasana', 'BaddhaKonasana', 'Downward_dog', 'Natarajasana', 'Triangle', 'UtkataKonasana', 'Veerabhadrasana', 'Vrukshasana']


,filepath,pose,landmark_path
0,/kaggle/input/datasets/kolliprasannaadarsh/man...,Vrukshasana,/kaggle/input/datasets/kolliprasannaadarsh/man...
1,/kaggle/input/datasets/kolliprasannaadarsh/man...,Vrukshasana,/kaggle/input/datasets/kolliprasannaadarsh/man...
2,/kaggle/input/datasets/kolliprasannaadarsh/man...,Vrukshasana,/kaggle/input/datasets/kolliprasannaadarsh/man...
3,/kaggle/input/datasets/kolliprasannaadarsh/man...,Vrukshasana,/kaggle/input/datasets/kolliprasannaadarsh/man...
4,/kaggle/input/datasets/kolliprasannaadarsh/man...,Vrukshasana,/kaggle/input/datasets/kolliprasannaadarsh/man...


In [7]:
# train val test split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["pose"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["pose"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 338
Validation: 73
Test: 73


## Pose Landmark Extraction

We use MediaPipe's PoseLandmarker (Tasks API) to extract 33 body keypoints from images. This replaces the deprecated `mp.solutions.pose` API.

In [9]:
!pip install mediapipe --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 84.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.


In [11]:
# This cell inspects the installed mediapipe modules.
import mediapipe as mp
print("mediapipe version:", mp.__version__)
print(dir(mp))

mediapipe version: 0.10.33
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [12]:
# Downloading the MediaPipe PoseLandmarker task model (Full/Medium) into 
# working directory for pose extraction
import os, urllib.request

os.makedirs("/kaggle/working/mp_models", exist_ok=True)

MODEL_PATH = "/kaggle/working/mp_models/pose_landmarker_full.task"
URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"

if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(URL, MODEL_PATH)

print("Model exists:", os.path.exists(MODEL_PATH), "->", MODEL_PATH)
print("Size (MB):", round(os.path.getsize(MODEL_PATH)/1024/1024, 2))

Model exists: True -> /kaggle/working/mp_models/pose_landmarker_full.task
Size (MB): 8.96


In [13]:
# This cell initializes MediaPipe Tasks PoseLandmarker for 
# single-person pose detection on static images.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
RunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.IMAGE,
    num_poses=1
)

landmarker = PoseLandmarker.create_from_options(options)
print("PoseLandmarker ready")

PoseLandmarker ready


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774129562.902887     167 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774129562.955797     167 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [14]:
# Test pose extraction on one image (also called a Smoke Test, to confirm that
# the detection returns 33 landmarks)
import cv2
import numpy as np

test_fp = df.iloc[0]["filepath"]  # df must exist
img_bgr = cv2.imread(test_fp)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
result = landmarker.detect(mp_image)

print("File:", test_fp)
print("Num poses detected:", len(result.pose_landmarks))

if len(result.pose_landmarks) > 0:
    print("Landmarks in pose 0:", len(result.pose_landmarks[0]))
    print("First landmark:", result.pose_landmarks[0][0])

File: /kaggle/input/datasets/kolliprasannaadarsh/manoj-dataset/Manoj-Dataset/Vrukshasana/Images/istockphoto-1403294529-612x612.jpg
Num poses detected: 1
Landmarks in pose 0: 33
First landmark: NormalizedLandmark(x=0.49481701850891113, y=0.19191178679466248, z=-0.3081483542919159, visibility=0.9999736547470093, presence=0.999976634979248, name=None)


W0000 00:00:1774129566.191546     168 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [15]:
# This cell defines keypoint extraction (33 landmarks + visibility) and normalizes 
# coordinates by hip-center + shoulder-width.
LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12
LEFT_HIP, RIGHT_HIP = 23, 24

def extract_keypoints_task_api(filepath):
    img_bgr = cv2.imread(filepath)
    if img_bgr is None:
        return None
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = landmarker.detect(mp_image)

    if len(result.pose_landmarks) == 0:
        return None

    lms = result.pose_landmarks[0]  # 33 landmarks
    # each landmark has x,y,z,visibility
    kp = np.array([[p.x, p.y, p.z, getattr(p, "visibility", 1.0)] for p in lms], dtype=np.float32)
    return kp

def normalize_keypoints(kp):
    kp = kp.copy()
    hip_center = (kp[LEFT_HIP, :3] + kp[RIGHT_HIP, :3]) / 2.0
    kp[:, :3] -= hip_center
    scale = np.linalg.norm(kp[LEFT_SHOULDER, :3] - kp[RIGHT_SHOULDER, :3]) + 1e-6
    kp[:, :3] /= scale
    return kp

In [16]:
# This cell measurse pose detection failure rate.
sample = train_df.sample(80, random_state=0)

ok, fail = 0, 0
for fp in sample["filepath"]:
    kp = extract_keypoints_task_api(fp)
    if kp is None:
        fail += 1
    else:
        ok += 1

print("OK:", ok, "Fail:", fail, "Fail%:", round(100*fail/(ok+fail), 2))

OK: 79 Fail: 1 Fail%: 1.25


In [17]:
# Builds flattened features X,y and saves .npy.
import os
import numpy as np

os.makedirs("/kaggle/working/features", exist_ok=True)

poses = sorted(df["pose"].unique())
label_to_id = {p:i for i,p in enumerate(poses)}

def build_Xy(split_df):
    X, y = [], []
    skipped = 0
    
    for _, row in split_df.iterrows():
        kp = extract_keypoints_task_api(row["filepath"])
        if kp is None:
            skipped += 1
            continue
        kp = normalize_keypoints(kp)
        X.append(kp.reshape(-1))              # 33*4 = 132
        y.append(label_to_id[row["pose"]])
        
    return np.array(X, np.float32), np.array(y, np.int64), skipped

X_train, y_train, s1 = build_Xy(train_df)
X_val, y_val, s2 = build_Xy(val_df)
X_test, y_test, s3 = build_Xy(test_df)

np.save("/kaggle/working/features/X_train.npy", X_train)
np.save("/kaggle/working/features/y_train.npy", y_train)
np.save("/kaggle/working/features/X_val.npy", X_val)
np.save("/kaggle/working/features/y_val.npy", y_val)
np.save("/kaggle/working/features/X_test.npy", X_test)
np.save("/kaggle/working/features/y_test.npy", y_test)

print("Train:", X_train.shape, "skipped:", s1)
print("Val:", X_val.shape, "skipped:", s2)
print("Test:", X_test.shape, "skipped:", s3)
print("Classes:", len(poses), poses)


Train: (336, 132) skipped: 2
Val: (73, 132) skipped: 0
Test: (72, 132) skipped: 1
Classes: 8 ['ArdhaChandrasana', 'BaddhaKonasana', 'Downward_dog', 'Natarajasana', 'Triangle', 'UtkataKonasana', 'Veerabhadrasana', 'Vrukshasana']


In [18]:
# This cell loads arrays and prints the shapes of train test and validation sets.
import numpy as np

X_train = np.load("/kaggle/working/features/X_train.npy")
y_train = np.load("/kaggle/working/features/y_train.npy")
X_val   = np.load("/kaggle/working/features/X_val.npy")
y_val   = np.load("/kaggle/working/features/y_val.npy")
X_test  = np.load("/kaggle/working/features/X_test.npy")
y_test  = np.load("/kaggle/working/features/y_test.npy")

print("Shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

num_classes = int(np.max(y_train)) + 1
print("num_classes:", num_classes)


Shapes:
Train: (336, 132) (336,)
Val  : (73, 132) (73,)
Test : (72, 132) (72,)
num_classes: 8


In [19]:
# This cell is used for building PyTorch loaders and for choosing the device = cuda.
import torch
from torch.utils.data import Dataset, DataLoader

class NumpyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(NumpyDataset(X_train, y_train), batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(NumpyDataset(X_val, y_val), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(NumpyDataset(X_test, y_test), batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla P100-PCIE-16GB


## Model Architecture
We use a Multi-Layer Perceptron (MLP) trained on normalized pose keypoints extracted using MediaPipe. The model learns to classify yoga poses based on spatial joint relationships.

In [20]:
# This cell defines MLP.
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, in_dim=132, num_classes=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model = MLP(in_dim=X_train.shape[1], num_classes=num_classes).to(device)
print(model)


MLP(
  (net): Sequential(
    (0): Linear(in_features=132, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=8, bias=True)
  )
)


In [21]:
# This cell is for the training loop (with an early stopping).
import torch
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_acc = 0.0
patience = 8
bad_epochs = 0

def eval_accuracy(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(Xb)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

for epoch in range(1, 51):
    model.train()
    running_loss = 0.0
    
    for Xb, yb in train_loader:
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * yb.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_acc = eval_accuracy(val_loader)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        bad_epochs = 0
        torch.save(model.state_dict(), "/kaggle/working/checkpoints_best_mlp.pt")
        print("  ✅ saved best checkpoint")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("  🛑 early stopping")
            break

print("Best val acc:", best_val_acc)


Epoch 01 | train_loss=2.0118 | val_acc=0.5616
  ✅ saved best checkpoint
Epoch 02 | train_loss=1.7716 | val_acc=0.5890
  ✅ saved best checkpoint
Epoch 03 | train_loss=1.5246 | val_acc=0.6575
  ✅ saved best checkpoint
Epoch 04 | train_loss=1.2781 | val_acc=0.6986
  ✅ saved best checkpoint
Epoch 05 | train_loss=1.0340 | val_acc=0.7534
  ✅ saved best checkpoint
Epoch 06 | train_loss=0.8423 | val_acc=0.8630
  ✅ saved best checkpoint
Epoch 07 | train_loss=0.6785 | val_acc=0.9041
  ✅ saved best checkpoint
Epoch 08 | train_loss=0.5818 | val_acc=0.9452
  ✅ saved best checkpoint
Epoch 09 | train_loss=0.4694 | val_acc=0.9452
Epoch 10 | train_loss=0.3709 | val_acc=0.9452
Epoch 11 | train_loss=0.3153 | val_acc=0.9315
Epoch 12 | train_loss=0.3013 | val_acc=0.9452
Epoch 13 | train_loss=0.2612 | val_acc=0.9452
Epoch 14 | train_loss=0.2288 | val_acc=0.9452
Epoch 15 | train_loss=0.1931 | val_acc=0.9452
Epoch 16 | train_loss=0.1810 | val_acc=0.9589
  ✅ saved best checkpoint
Epoch 17 | train_loss=0.1604 |

## Model Evaluation
We evaluate the trained model using multiple metrics including accuracy, F1-score, precision, recall, and confusion matrix analysis to understand class-wise performance.

In [22]:
# This cell contains the evaluation metrics like Test accuracy, confusion matrix, etc.
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# load best model
model.load_state_dict(torch.load("/kaggle/working/checkpoints_best_mlp.pt", map_location=device))

# predict on test
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device, non_blocking=True)
        logits = model(Xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

test_acc = (all_preds == all_true).mean()
print("Test accuracy:", test_acc)

cm = confusion_matrix(all_true, all_preds)
print("Confusion matrix:\n", cm)

print("\nClassification report:\n")
print(classification_report(all_true, all_preds))


Test accuracy: 0.9861111111111112
Confusion matrix:
 [[9 0 0 0 0 0 0 0]
 [0 9 0 0 0 0 0 0]
 [0 0 9 0 0 0 0 0]
 [0 0 0 9 0 0 0 0]
 [0 0 0 0 9 0 0 0]
 [0 0 0 1 0 9 0 0]
 [0 0 0 0 0 0 9 0]
 [0 0 0 0 0 0 0 8]]

Classification report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00         9
           3       0.90      1.00      0.95         9
           4       1.00      1.00      1.00         9
           5       1.00      0.90      0.95        10
           6       1.00      1.00      1.00         9
           7       1.00      1.00      1.00         8

    accuracy                           0.99        72
   macro avg       0.99      0.99      0.99        72
weighted avg       0.99      0.99      0.99        72



In [23]:
# This cell saves the unflattened keypoints.
import numpy as np
import os

os.makedirs("/kaggle/working/features", exist_ok=True)

def build_Ky(split_df):
    K, y = [], []
    skipped = 0
    for _, row in split_df.iterrows():
        kp = extract_keypoints_task_api(row["filepath"])
        if kp is None:
            skipped += 1
            continue
        kp = normalize_keypoints(kp)          # (33,4)
        K.append(kp)
        y.append(label_to_id[row["pose"]])
    return np.array(K, np.float32), np.array(y, np.int64), skipped

K_train, y_train2, s1 = build_Ky(train_df)
K_val,   y_val2,   s2 = build_Ky(val_df)
K_test,  y_test2,  s3 = build_Ky(test_df)

np.save("/kaggle/working/features/K_train.npy", K_train)
np.save("/kaggle/working/features/K_val.npy", K_val)
np.save("/kaggle/working/features/K_test.npy", K_test)

print("K_train:", K_train.shape, "skipped:", s1)
print("K_val:", K_val.shape, "skipped:", s2)
print("K_test:", K_test.shape, "skipped:", s3)

K_train: (336, 33, 4) skipped: 2
K_val: (73, 33, 4) skipped: 0
K_test: (72, 33, 4) skipped: 1


## Pose Quality Feedback System
We compute joint angles from pose landmarks and compare them against reference distributions to generate corrective feedback for improving pose alignment.

In [24]:
# This cell defines helper to compute the joint angles from 2D points.
import numpy as np

def angle_3pts_2d(a, b, c):
    """
    angle at point b using only x,y (ignore z)
    a,b,c: shape (2,)
    returns degrees
    """
    ba = a - b
    bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    cosang = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return np.degrees(np.arccos(cosang))


In [25]:
# This cell defines the angle based keypoints (elbows, knees, hips)
LS, RS = 11, 12
LE, RE = 13, 14
LW, RW = 15, 16
LH, RH = 23, 24
LK, RK = 25, 26
LA, RA = 27, 28

def angle_features(kp33x4, vis_thresh=0.6):
    xy = kp33x4[:, :2]   # ONLY x,y
    vis = kp33x4[:, 3]

    def safe_angle(a, b, c):
        if min(vis[a], vis[b], vis[c]) < vis_thresh:
            return None
        return angle_3pts_2d(xy[a], xy[b], xy[c])

    feats = {
        "left_elbow":  safe_angle(LS, LE, LW),
        "right_elbow": safe_angle(RS, RE, RW),
        "left_knee":   safe_angle(LH, LK, LA),
        "right_knee":  safe_angle(RH, RK, RA),
        "left_hip":    safe_angle(LS, LH, LK),
        "right_hip":   safe_angle(RS, RH, RK),
    }

    return {k:v for k,v in feats.items() if v is not None}


In [26]:
# This cell builds the per pose reference angle distributions.
import numpy as np

K_train = np.load("/kaggle/working/features/K_train.npy")
y_train_used = y_train  # assumes aligned with K_train

KEYS = ["left_elbow","right_elbow","left_knee","right_knee","left_hip","right_hip"]
VIS_THRESH = 0.3        # <- lowered

pose_refs = {}

for pose_id in range(len(poses)):
    idx = np.where(y_train_used == pose_id)[0]
    if len(idx) < 10:
        continue

    # collect vals per key independently
    collected = {k: [] for k in KEYS}

    for i in idx:
        feats = angle_features(K_train[i], vis_thresh=VIS_THRESH)
        for k in KEYS:
            if k in feats:
                collected[k].append(feats[k])

    # only keep keys with enough samples
    ref = {}
    for k in KEYS:
        vals = np.array(collected[k], dtype=np.float32)
        if len(vals) >= 30:   # require at least 30 samples for stability
            ref[k] = (float(vals.mean()), float(vals.std() + 1e-6))

    if len(ref) >= 4:  # require at least 4 angles for this pose
        pose_refs[pose_id] = ref
    else:
        print("Skipping", poses[pose_id], "- usable angles:", list(ref.keys()), 
              "| counts:", {k: len(collected[k]) for k in KEYS})

print("Built refs for", len(pose_refs), "poses")


Skipping Downward_dog - usable angles: ['right_elbow', 'right_knee', 'right_hip'] | counts: {'left_elbow': 12, 'right_elbow': 42, 'left_knee': 9, 'right_knee': 42, 'left_hip': 9, 'right_hip': 42}
Built refs for 7 poses


In [27]:
# Turns the z-scores in friendly feedback text.
FRIENDLY = {
    "left_knee": "Left knee",
    "right_knee": "Right knee",
    "left_hip": "Left hip",
    "right_hip": "Right hip",
    "left_elbow": "Left elbow",
    "right_elbow": "Right elbow",
}

def feedback_for_pose_friendly(kp, pose_id, z_thresh=1.0, vis_thresh=0.3):
    feats = angle_features(kp, vis_thresh=vis_thresh)
    ref = pose_refs.get(pose_id, None)
    if ref is None:
        return ["No reference for this pose yet."]

    msgs = []
    for k, val in feats.items():
        if k not in ref:      
            continue

        mu, sigma = ref[k]
        z = (val - mu) / sigma
        name = FRIENDLY.get(k, k.replace("_", " "))

        if z > z_thresh:
            msgs.append(f"{name}: bend ~{(val-mu):.1f}° more.")
        elif z < -z_thresh:
            msgs.append(f"{name}: straighten ~{(mu-val):.1f}°.")

    if not msgs:
        msgs.append("Looks close to ideal for this pose.")
    return msgs


In [28]:
# Same feedback but this cell ignores the absurd deltas.
def feedback_for_pose_friendly_clipped(kp, pose_id, z_thresh=1.0, vis_thresh=0.3, max_delta=35.0):
    feats = angle_features(kp, vis_thresh=vis_thresh)
    ref = pose_refs.get(pose_id, None)
    if ref is None:
        return ["No reference for this pose yet."]

    msgs = []
    ignored = []

    for k, val in feats.items():
        if k not in ref:
            continue

        mu, sigma = ref[k]
        z = (val - mu) / sigma
        name = FRIENDLY.get(k, k.replace("_", " "))

        if abs(z) > z_thresh:
            delta = abs(val - mu)

            if delta > max_delta:
                ignored.append(f"{name} (delta {delta:.1f}°)")
                continue

            if val > mu:
                msgs.append(f"{name}: bend ~{delta:.1f}° more.")
            else:
                msgs.append(f"{name}: straighten ~{delta:.1f}°.")

    if not msgs:
        msgs.append("Looks close to ideal for this pose.")

    if ignored:
        msgs.append("Note: ignored unreliable joints: " + ", ".join(ignored))

    return msgs


In [29]:
# Demo Feedback at different thresholds.
i = 0
pose_id = int(y_test[i])
print("Pose:", poses[pose_id])

for zt in [1.5, 1.0, 0.8]:
    print("\n--- z_thresh =", zt, "---")
    msgs = feedback_for_pose_friendly(K_test[i], pose_id, z_thresh=zt, vis_thresh=0.3)
    for m in msgs[:10]:
        print("-", m)

Pose: Triangle

--- z_thresh = 1.5 ---
- Looks close to ideal for this pose.

--- z_thresh = 1.0 ---
- Left knee: straighten ~4.5°.

--- z_thresh = 0.8 ---
- Right elbow: bend ~3.9° more.
- Left knee: straighten ~4.5°.
- Right knee: bend ~3.8° more.


In [30]:
# Finds the worst test sample and prints it's feedback.
def worst_score_clipped(kp, pose_id, vis_thresh=0.3, max_delta=35.0):
    feats = angle_features(kp, vis_thresh=vis_thresh)
    ref = pose_refs.get(pose_id, None)
    if ref is None:
        return 0.0

    worst = 0.0
    for k, val in feats.items():
        if k not in ref:
            continue

        mu, sigma = ref[k]
        delta = abs(val - mu)

        # ignore absurd deltas (likely landmark failure)
        if delta > max_delta:
            continue

        z = abs((val - mu) / sigma)
        worst = max(worst, z)

    return worst


# find worst test sample (consistent with clipped feedback)
worst_i = None
worst_score = -1

for i in range(len(K_test)):
    pid = int(y_test[i])
    score = worst_score_clipped(K_test[i], pid, vis_thresh=0.3, max_delta=35.0)
    if score > worst_score:
        worst_score = score
        worst_i = i

pid = int(y_test[worst_i])
print("Worst i:", worst_i, "score:", worst_score, "pose:", poses[pid])

msgs = feedback_for_pose_friendly_clipped(K_test[worst_i], pid, z_thresh=1.0, vis_thresh=0.3, max_delta=35.0)
for m in msgs[:10]:
    print("-", m)


Worst i: 35 score: 2.7989764 pose: Veerabhadrasana
- Left elbow: straighten ~11.5°.
- Note: ignored unreliable joints: Left knee (delta 42.9°), Right knee (delta 35.3°)


## What the previous code does:

Up to now, we built a pose classifier (MLP) and then built "ideal pose" references from the training set (mean/std of joint angles per pose).

The last section uses those references to **scan the test set** and find the single "worst" example:
- For each test image, compute joint angles from its keypoints
- Compare angles to the training-derived reference for that pose (via z-scores)
- Pick the test image with the largest deviation (largest |z| after clipping unreliable joints)
This is mainly a **debug/demo step** to inspect the most misaligned example and see the feedback text.

## What the new code below will achieve

Next, we add an inference function that works on any input image you provide:
1) Extract MediaPipe keypoints from the image and normalize them
2) Use the trained MLP to **predict the pose label** (with confidence)
3) Estimate **pose quality** by comparing the image to training references:
   - Angle-based deviation (human-readable feedback)
   - Keypoint deviation (which landmarks are farthest from the "ideal" landmark positions for that pose)
So the new function gives you: predicted pose + confidence + "what to fix" feedback + a ranked list of the most-off keypoints. (Previous correction was just like a test to check what would work and how it would work and to in a sense, test the model. This next portion actually takes the image, classifies it into one of the 9 poses and corrects any minor errors present).


In [31]:
#  Building per-pose landmark reference (mean/std) from training keypoints 
import numpy as np

# MediaPipe Pose has 33 landmarks. Names help interpret which keypoint is "off".
MP_LANDMARK_NAMES = [
    "nose","left_eye_inner","left_eye","left_eye_outer","right_eye_inner","right_eye","right_eye_outer",
    "left_ear","right_ear","mouth_left","mouth_right",
    "left_shoulder","right_shoulder","left_elbow","right_elbow","left_wrist","right_wrist",
    "left_pinky","right_pinky","left_index","right_index","left_thumb","right_thumb",
    "left_hip","right_hip","left_knee","right_knee","left_ankle","right_ankle",
    "left_heel","right_heel","left_foot_index","right_foot_index"
]

def build_landmark_refs(K_train, y_train_used, vis_thresh=0.3, min_samples=30, eps=1e-6):
    
    landmark_refs = {}
    num_poses = int(np.max(y_train_used)) + 1
    
    for pose_id in range(num_poses):
        idx = np.where(y_train_used == pose_id)[0]
        if len(idx) < 10:
            continue
        
        # store per-landmark xyz samples
        collected = [[] for _ in range(33)]
        
        for i in idx:
            kp = K_train[i]  # (33,4) normalized already in your pipeline
            xyz = kp[:, :3]
            vis = kp[:, 3]
            for j in range(33):
                if vis[j] >= vis_thresh:
                    collected[j].append(xyz[j])
        
        mu = np.zeros((33, 3), dtype=np.float32)
        sigma = np.ones((33, 3), dtype=np.float32)
        count = np.zeros((33,), dtype=np.int32)
        
        for j in range(33):
            vals = np.array(collected[j], dtype=np.float32)
            count[j] = len(vals)
            if len(vals) >= min_samples:
                mu[j] = vals.mean(axis=0)
                sigma[j] = vals.std(axis=0) + eps
            else:
                # Not enough stable samples: keep default sigma=1 to avoid division blowups.
                # mu remains 0 which is reasonable because your normalization centers around hips.
                pass
        
        landmark_refs[pose_id] = {"mu": mu, "sigma": sigma, "count": count}
    
    return landmark_refs

# Build landmark references using your already-saved/loaded K_train
# NOTE: this uses y_train_used = y_train in your earlier cell, which assumes alignment with K_train
landmark_refs = build_landmark_refs(K_train, y_train_used=y_train, vis_thresh=0.3, min_samples=30)
print("Built landmark refs for", len(landmark_refs), "poses")


Built landmark refs for 8 poses


In [32]:
import torch
import numpy as np

def predict_pose_from_kp_flat(kp_flat, model, device, poses):

    model.eval()
    x = torch.from_numpy(kp_flat.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred_id = int(np.argmax(probs))
    return pred_id, poses[pred_id], float(probs[pred_id]), probs

def keypoint_deviation_report(kp33x4, pose_id, landmark_refs, top_k=8, vis_thresh=0.3):
    """
    For a given pose_id what we're doing is:
    - compare each landmark xyz to pose mean xyz (in normalized coords)
    - compute a z-like magnitude per landmark (using per-landmark std)
    which returns a sorted list of dicts for the worst landmarks
    """
    ref = landmark_refs.get(pose_id, None)
    if ref is None:
        return []

    xyz = kp33x4[:, :3]
    vis = kp33x4[:, 3]

    mu = ref["mu"]
    sigma = ref["sigma"]

    items = []
    for j in range(33):
        if vis[j] < vis_thresh:
            continue

        delta = xyz[j] - mu[j]                 # (3,)
        # z per-dimension; then aggregate into one magnitude number for ranking
        z = delta / sigma[j]                   # (3,)
        zmag = float(np.linalg.norm(z))
        dmag = float(np.linalg.norm(delta))

        items.append({
            "landmark_id": j,
            "landmark_name": MP_LANDMARK_NAMES[j],
            "delta_xyz": delta,
            "delta_norm": dmag,   # in normalized units (scaled by shoulder width)
            "z_mag": zmag,
            "visibility": float(vis[j]),
        })

    items.sort(key=lambda x: x["z_mag"], reverse=True)
    return items[:top_k]

def analyze_image_pose_and_quality(
    image_path,
    model,
    device,
    poses,
    pose_refs,
    landmark_refs,
    z_thresh=1.0,
    vis_thresh=0.3,
    max_delta_deg=35.0,
    top_k_landmarks=8
):
    """
    End-to-end Process of what we're planning on doing:
      image -> keypoints -> predict pose -> angle feedback + keypoint deviation ranking
    Returns a dict we can print/log.
    """
    # 1) Extract + normalize keypoints
    kp = extract_keypoints_task_api(image_path)
    if kp is None:
        return {"ok": False, "error": "No pose detected or failed to read image.", "image_path": image_path}

    kp = normalize_keypoints(kp)  # (33,4)

    # 2) Predict pose using MLP
    kp_flat = kp.reshape(-1)  # (132,)
    pred_id, pred_name, conf, probs = predict_pose_from_kp_flat(kp_flat, model, device, poses)

    # 3) Angle-based coaching feedback (compares angles to training pose_refs)
    angle_msgs = feedback_for_pose_friendly_clipped(
        kp, pred_id,
        z_thresh=z_thresh,
        vis_thresh=vis_thresh,
        max_delta=max_delta_deg
    )

    # 4) Keypoint deviation report (compares landmark xyz to training landmark template)
    worst_landmarks = keypoint_deviation_report(
        kp, pred_id,
        landmark_refs=landmark_refs,
        top_k=top_k_landmarks,
        vis_thresh=vis_thresh
    )

    return {
        "ok": True,
        "image_path": image_path,
        "pred_pose_id": pred_id,
        "pred_pose_name": pred_name,
        "confidence": conf,
        "angle_feedback": angle_msgs,
        "worst_landmarks": worst_landmarks,
    }



model.load_state_dict(torch.load("/kaggle/working/checkpoints_best_mlp.pt", map_location=device))
model.to(device)
model.eval()

print("Ready: analyze_image_pose_and_quality(image_path, ...)")


Ready: analyze_image_pose_and_quality(image_path, ...)


In [34]:
# Example: pass any image path from the dataframe or any other image from your laptop.
example_fp = df.iloc[222]["filepath"]

result = analyze_image_pose_and_quality(
    image_path=example_fp,
    model=model,
    device=device,
    poses=poses,
    pose_refs=pose_refs,
    landmark_refs=landmark_refs,
    z_thresh=1.0,
    vis_thresh=0.3,
    max_delta_deg=35.0,
    top_k_landmarks=8
)

if not result["ok"]:
    print("Error:", result["error"])
else:
    print("Predicted pose:", result["pred_pose_name"], "| confidence:", round(result["confidence"], 4))
    print("\nAngle feedback:")
    for m in result["angle_feedback"]:
        print("-", m)

    print("\nMost-off landmarks (ranked):")
    for item in result["worst_landmarks"]:
        print(f"- {item['landmark_name']}: z_mag={item['z_mag']:.2f}, delta_norm={item['delta_norm']:.3f}, vis={item['visibility']:.2f}")


Predicted pose: BaddhaKonasana | confidence: 0.9977

Angle feedback:
- Left hip: straighten ~21.8°.

Most-off landmarks (ranked):
- left_shoulder: z_mag=1.58, delta_norm=0.379, vis=0.99
- left_ear: z_mag=1.48, delta_norm=0.674, vis=1.00
- left_knee: z_mag=1.41, delta_norm=0.394, vis=0.96
- right_elbow: z_mag=1.39, delta_norm=0.276, vis=0.88
- left_eye_outer: z_mag=1.36, delta_norm=0.713, vis=1.00
- mouth_left: z_mag=1.34, delta_norm=0.640, vis=1.00
- left_eye: z_mag=1.33, delta_norm=0.712, vis=1.00
- mouth_right: z_mag=1.33, delta_norm=0.631, vis=1.00
